In [1]:
%load_ext autoreload
%autoreload 2


**Author:** Salvador Navas  
**Date:** 2025-06-27

In [2]:
from pyhydra.data_sources.rainfall import OGIMETDownloader, OgimetCSVLoader
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
import tempfile, os


# OGIMET Data Download & Analysis

[OGIMET](https://www.ogimet.com) provides historical weather observations from WMO/SYNOP stations worldwide.

## Workflow

1. **Download**: Use `OGIMETDownloader(stations_csv)` — interactive Jupyter widget that shows a map.
   Users click on stations and select a date range to download observation CSVs.
2. **Load**: `OgimetCSVLoader(folder)` reads the downloaded CSVs.
3. **Analyse**: Quality metrics, missing data, visualisation.

```python
# Real usage (requires a stations CSV file and Jupyter widgets):
# from pyhydra.data_sources.rainfall import OGIMETDownloader
# OGIMETDownloader("path/to/stations.csv")   # opens interactive map
```

---
## Demo with synthetic OGIMET-format data

This notebook generates synthetic station data in the exact format produced by `OGIMETDownloader` so the loading and analysis API can be demonstrated without a real OGIMET download.


In [3]:
# ── Create a temporary folder with synthetic OGIMET-format files ───────────
DEMO_DIR = Path(tempfile.mkdtemp(prefix="ogimet_demo_"))

rng = np.random.default_rng(42)
station_ids = ["08181", "08221", "08304", "08307"]
station_meta = [
    {"station_id": "08181", "name": "Madrid-Retiro",   "lat": 40.41, "lon": -3.68, "elev": 667},
    {"station_id": "08221", "name": "Toledo",           "lat": 39.88, "lon": -4.05, "elev": 515},
    {"station_id": "08304", "name": "Seville",          "lat": 37.42, "lon": -5.90, "elev": 34},
    {"station_id": "08307", "name": "Badajoz",          "lat": 38.88, "lon": -6.82, "elev": 185},
]
dates = pd.date_range("2010-01-01", "2020-12-31", freq="D")

for meta in station_meta:
    # station CSV (metadata)
    pd.DataFrame([meta]).to_csv(DEMO_DIR / f"station_{meta['station_id']}.csv", index=False)
    
    # series CSV (daily precipitation)
    seasonal = 2.5 + 2.0 * np.sin(2 * np.pi * (np.asarray(dates.dayofyear, float) / 365 - 0.1))
    wet = rng.random(len(dates)) < (0.25 + 0.1 * seasonal / seasonal.max())
    precip = np.where(wet, rng.exponential(seasonal, len(dates)), 0.0)
    # add some missing values
    mask = rng.random(len(dates)) < 0.02
    precip[mask] = np.nan
    tmax = 20 + 10 * np.sin(2 * np.pi * (np.asarray(dates.dayofyear, float) / 365 - 0.25)) + rng.normal(0, 3, len(dates))
    df = pd.DataFrame({
        "date": dates,
        "precipitation_mm": precip.round(1),
        "tmax_celsius": tmax.round(1),
    })
    df.to_csv(DEMO_DIR / f"series_{meta['station_id']}.csv", index=False)

print(f"Demo data written to: {DEMO_DIR}")
print(f"Files: {[f.name for f in DEMO_DIR.iterdir()]}")


Demo data written to: /var/folders/44/dw7p6q9108xcc4mmh_f7q1vc0000gn/T/ogimet_demo_ed046i6l
Files: ['station_08181.csv', 'station_08221.csv', 'series_08304.csv', 'series_08307.csv', 'series_08181.csv', 'series_08221.csv', 'station_08307.csv', 'station_08304.csv']


In [4]:
# ── Load station metadata ───────────────────────────────────────────────────
loader = OgimetCSVLoader(str(DEMO_DIR))
stations_df = loader.load_station_data()
print(f"Loaded {len(stations_df)} stations:")
print(stations_df.to_string(index=False))


Loaded 4 stations:
 station_id          name   lat   lon  elev
       8181 Madrid-Retiro 40.41 -3.68   667
       8221        Toledo 39.88 -4.05   515
       8307       Badajoz 38.88 -6.82   185
       8304       Seville 37.42 -5.90    34


In [5]:
# ── Load precipitation series ────────────────────────────────────────────────
series_df = loader.load_series_data("precipitation_mm")
print(f"Series shape: {series_df.shape}  (days × stations)")
print(f"Period: {series_df.index[0].date()} → {series_df.index[-1].date()}")
print("\nFirst 5 rows:")
print(series_df.head())


Series shape: (4018, 4)  (days × stations)
Period: 2010-01-01 → 2020-12-31

First 5 rows:
            08304  08307  08181  08221
date                                  
2010-01-01    0.4    0.7    0.0    0.0
2010-01-02    0.0    0.0    0.0    0.0
2010-01-03    0.6    0.0    0.0    1.0
2010-01-04    0.0    0.0    0.0    0.9
2010-01-05    0.0    0.0    1.2    0.7


In [6]:
# ── Quality analysis ─────────────────────────────────────────────────────────
quality_df = loader.analyze_series_quality()
print("Quality metrics per station:")
print(quality_df.to_string(index=False))


Quality metrics per station:
station_id  start_year  end_year  missing_percent  full_years  full_months  min_value  max_value
     08304        2010      2020             1.79           0          129        0.0       29.2
     08307        2010      2020             1.92           0          128        0.0       22.7
     08181        2010      2020             1.97           0          126        0.0       36.8
     08221        2010      2020             1.79           0          127        0.0       37.7


In [7]:
# ── Visualise ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 7))
axes = axes.flatten()

for ax, col in zip(axes, series_df.columns):
    s = series_df[col].dropna()
    monthly = s.resample("ME").sum()
    ax.bar(monthly.index, monthly.values, width=25, color="steelblue", alpha=0.7)
    ax.set_title(col, fontsize=10)
    ax.set_ylabel("Monthly precip (mm)")
    ax.grid(alpha=0.3, axis="y")

plt.suptitle("Monthly precipitation — OGIMET stations (synthetic demo)", fontsize=12)
plt.tight_layout()
plt.savefig("/tmp/ogimet_precip.png", dpi=100)
plt.close()
print("Monthly precipitation plot saved to /tmp/ogimet_precip.png")
print(f"\nAnnual precipitation (mean over stations): {series_df.resample('YE').sum().mean(axis=1).round(0).to_string()}")


Monthly precipitation plot saved to /tmp/ogimet_precip.png

Annual precipitation (mean over stations): date
2010-12-31    289.0
2011-12-31    292.0
2012-12-31    300.0
2013-12-31    293.0
2014-12-31    289.0
2015-12-31    299.0
2016-12-31    267.0
2017-12-31    266.0
2018-12-31    270.0
2019-12-31    284.0
2020-12-31    321.0
Freq: YE-DEC
